In [1]:
# IMPORTS 
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px

In [2]:
# load in data
df = pd.read_csv("medicare_drug_spending.csv")
df.shape

(14309, 46)

In [ ]:
df.columns

Index(['Brnd_Name', 'Gnrc_Name', 'Tot_Mftr', 'Mftr_Name', 'Tot_Spndng_2019',
       'Tot_Dsg_Unts_2019', 'Tot_Clms_2019', 'Tot_Benes_2019',
       'Avg_Spnd_Per_Dsg_Unt_Wghtd_2019', 'Avg_Spnd_Per_Clm_2019',
       'Avg_Spnd_Per_Bene_2019', 'Outlier_Flag_2019', 'Tot_Spndng_2020',
       'Tot_Dsg_Unts_2020', 'Tot_Clms_2020', 'Tot_Benes_2020',
       'Avg_Spnd_Per_Dsg_Unt_Wghtd_2020', 'Avg_Spnd_Per_Clm_2020',
       'Avg_Spnd_Per_Bene_2020', 'Outlier_Flag_2020', 'Tot_Spndng_2021',
       'Tot_Dsg_Unts_2021', 'Tot_Clms_2021', 'Tot_Benes_2021',
       'Avg_Spnd_Per_Dsg_Unt_Wghtd_2021', 'Avg_Spnd_Per_Clm_2021',
       'Avg_Spnd_Per_Bene_2021', 'Outlier_Flag_2021', 'Tot_Spndng_2022',
       'Tot_Dsg_Unts_2022', 'Tot_Clms_2022', 'Tot_Benes_2022',
       'Avg_Spnd_Per_Dsg_Unt_Wghtd_2022', 'Avg_Spnd_Per_Clm_2022',
       'Avg_Spnd_Per_Bene_2022', 'Outlier_Flag_2022', 'Tot_Spndng_2023',
       'Tot_Dsg_Unts_2023', 'Tot_Clms_2023', 'Tot_Benes_2023',
       'Avg_Spnd_Per_Dsg_Unt_Wghtd_2023', 'Avg_S

## Data Cleaning
1. Clean up year column and expand out rows 
1. Merge atc_disease_mapping to medicare_drug_spending data
1. Seperate overall from manufacturer-specifc data 
1. Calculate derived average price column by condition
1. Drop rows with `Outlier_Flag` set as True
1. Remove '*' from `Mftr_Name`

In [4]:
id_vars = ['Brnd_Name', 'Gnrc_Name', 'Tot_Mftr', 'Mftr_Name']

# use wide_to_long to reshape
df_long = pd.wide_to_long(
    df,
    stubnames=[
        'Avg_Spnd_Per_Bene',
        'Avg_Spnd_Per_Clm',
        'Avg_Spnd_Per_Dsg_Unt_Wghtd',
        'Outlier_Flag',
        'Tot_Benes',
        'Tot_Clms',
        'Tot_Dsg_Unts',
        'Tot_Spndng'
    ],
    i=id_vars,          # columns that identify unique rows
    j='Year',           # new column name for year
    sep='_',            # separator between stub and year
    suffix='\d{4}'      # pattern for years
).reset_index()

# convert to categorical variable
df_long["Outlier_Flag"] = df_long["Outlier_Flag"].astype(str)



<>:19: SyntaxWarning: invalid escape sequence '\d'
<>:19: SyntaxWarning: invalid escape sequence '\d'
/var/folders/5b/6lvktnb96955q0486jnqvwf80000gn/T/ipykernel_74899/2484008443.py:19: SyntaxWarning: invalid escape sequence '\d'
  suffix='\d{4}'      # pattern for years


In [5]:
df_long.shape

(71545, 15)

In [6]:
df_long.dtypes

Brnd_Name                           object
Gnrc_Name                           object
Tot_Mftr                             int64
Mftr_Name                           object
Year                                 int64
CAGR_Avg_Spnd_Per_Dsg_Unt_19_23    float64
Chg_Avg_Spnd_Per_Dsg_Unt_22_23     float64
Avg_Spnd_Per_Bene                  float64
Avg_Spnd_Per_Clm                   float64
Avg_Spnd_Per_Dsg_Unt_Wghtd         float64
Outlier_Flag                        object
Tot_Benes                          float64
Tot_Clms                           float64
Tot_Dsg_Unts                       float64
Tot_Spndng                         float64
dtype: object

In [7]:
# Load mapping file
mapping = pd.read_csv("atc_disease_mapping.csv")

mapping.head()

,Gnrc_Name,ATC_Codes,Disease_Category
0,0.9 % Sodium Chloride,NaN,NaN
1,Aa 5 %/Calcium/Lytes/Dext 20 %,NaN,NaN
2,Aa 5%/D15w/Electrolytes,NaN,NaN
3,Abacavir Sulfate,J05AF,NaN
4,Abacavir Sulfate/Lamivudine,NaN,NaN


In [12]:
# 3. Merge on Gnrc_Name (generic name)
medicare_with_cond = df_long.merge(
    mapping[["Gnrc_Name", "ATC_Codes", "Disease_Category"]],
    on="Gnrc_Name",
    how="left"    # keep all Medicare rows, add condition info where available
)

# 4. Make blanks explicit by labelling as "Other / Not classified"
medicare_with_cond["Disease_Category"] = (
    medicare_with_cond["Disease_Category"]
    .fillna("Other / Not classified")
)

medicare_with_cond

,Brnd_Name,Gnrc_Name,Tot_Mftr,Mftr_Name,Year,CAGR_Avg_Spnd_Per_Dsg_Unt_19_23,Chg_Avg_Spnd_Per_Dsg_Unt_22_23,Avg_Spnd_Per_Bene,Avg_Spnd_Per_Clm,Avg_Spnd_Per_Dsg_Unt_Wghtd,Outlier_Flag,Tot_Benes,Tot_Clms,Tot_Dsg_Unts,Tot_Spndng,ATC_Codes,Disease_Category
0,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Overall,2019,0.011754,0.005702,74.122300,25.816335,0.216788,0.0,1878.0,5392.0,642471.0,139201.68,NaN,Other / Not classified
1,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Overall,2020,0.011754,0.005702,74.560025,26.682351,0.217701,0.0,1595.0,4457.0,547006.0,118923.24,NaN,Other / Not classified
2,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Overall,2021,0.011754,0.005702,77.898522,27.583808,0.223001,0.0,1313.0,3708.0,459384.0,102280.76,NaN,Other / Not classified
3,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Overall,2022,0.011754,0.005702,61.063304,28.004642,0.225874,0.0,1147.0,2501.0,310304.0,70039.61,NaN,Other / Not classified
4,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Overall,2023,0.011754,0.005702,63.454993,27.498475,0.227162,0.0,699.0,1613.0,195672.0,44355.04,NaN,Other / Not classified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71540,Zyvox,Linezolid In Dextrose 5%,1,Phar-Prep/Pfize,2019,-0.030330,-0.086898,956.141500,434.609773,0.173214,0.0,20.0,44.0,110400.0,19122.83,NaN,Other / Not classified
71541,Zyvox,Linezolid In Dextrose 5%,1,Phar-Prep/Pfize,2020,-0.030330,-0.086898,851.452400,332.598594,0.164246,0.0,25.0,64.0,129600.0,21286.31,NaN,Other / Not classified
71542,Zyvox,Linezolid In Dextrose 5%,1,Phar-Prep/Pfize,2021,-0.030330,-0.086898,NaN,257.684324,0.160510,0.0,NaN,37.0,59400.0,9534.32,NaN,Other / Not classified
71543,Zyvox,Linezolid In Dextrose 5%,1,Phar-Prep/Pfize,2022,-0.030330,-0.086898,1073.345833,368.004286,0.167710,1.0,12.0,35.0,76800.0,12880.15,NaN,Other / Not classified


In [13]:
# filter out for just the overall data for manufacturers 
medicare_overall = medicare_with_cond[medicare_with_cond['Mftr_Name'] == 'Overall']

# get rid of outlier flag data? 
medicare_overall = medicare_overall.drop(
    medicare_overall[medicare_overall['Outlier_Flag'] == '1.0'].index
) 

medicare_overall

,Brnd_Name,Gnrc_Name,Tot_Mftr,Mftr_Name,Year,CAGR_Avg_Spnd_Per_Dsg_Unt_19_23,Chg_Avg_Spnd_Per_Dsg_Unt_22_23,Avg_Spnd_Per_Bene,Avg_Spnd_Per_Clm,Avg_Spnd_Per_Dsg_Unt_Wghtd,Outlier_Flag,Tot_Benes,Tot_Clms,Tot_Dsg_Unts,Tot_Spndng,ATC_Codes,Disease_Category
0,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Overall,2019,0.011754,0.005702,74.122300,25.816335,0.216788,0.0,1878.0,5392.0,642471.0,139201.68,NaN,Other / Not classified
1,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Overall,2020,0.011754,0.005702,74.560025,26.682351,0.217701,0.0,1595.0,4457.0,547006.0,118923.24,NaN,Other / Not classified
2,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Overall,2021,0.011754,0.005702,77.898522,27.583808,0.223001,0.0,1313.0,3708.0,459384.0,102280.76,NaN,Other / Not classified
3,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Overall,2022,0.011754,0.005702,61.063304,28.004642,0.225874,0.0,1147.0,2501.0,310304.0,70039.61,NaN,Other / Not classified
4,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Overall,2023,0.011754,0.005702,63.454993,27.498475,0.227162,0.0,699.0,1613.0,195672.0,44355.04,NaN,Other / Not classified
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
71530,Zyvox,Linezolid In Dextrose 5%,2,Overall,2019,-0.066168,-0.050758,1108.234286,576.062376,0.205186,0.0,105.0,202.0,564610.0,116364.60,NaN,Other / Not classified
71531,Zyvox,Linezolid In Dextrose 5%,2,Overall,2020,-0.066168,-0.050758,833.546292,388.406387,0.163897,0.0,89.0,191.0,452706.0,74185.62,NaN,Other / Not classified
71532,Zyvox,Linezolid In Dextrose 5%,2,Overall,2021,-0.066168,-0.050758,871.001607,306.767862,0.163115,0.0,56.0,159.0,298817.0,48776.09,NaN,Other / Not classified
71533,Zyvox,Linezolid In Dextrose 5%,2,Overall,2022,-0.066168,-0.050758,817.559474,361.247209,0.164379,0.0,38.0,86.0,189002.0,31067.26,NaN,Other / Not classified


In [25]:
# export out to cleaned CSV file to read into Altair plot
medicare_overall.to_csv('medicare_with_conditions.csv')

In [14]:
# get average price per condition 
medicare_conditon_agg = medicare_overall.groupby(['Year', 'Disease_Category']).agg(
    Avg_Price_By_Cond=('Avg_Spnd_Per_Dsg_Unt_Wghtd', 'mean'),
    Total_Spend_By_Cond=('Tot_Spndng', 'sum')
).reset_index()

# create csv for average price by year 
medicare_conditon_agg 

,Year,Disease_Category,Avg_Price_By_Cond,Total_Spend_By_Cond
0,2019,Arthritis,1543.157217,2.198443e+10
1,2019,Arthritis;Hypertension,6.324735,2.472512e+08
2,2019,Diabetes,59.982048,1.766225e+10
3,2019,Diabetes;High Cholesterol,0.138261,2.069756e+08
4,2019,High Cholesterol,122.985532,3.091740e+09
5,2019,Hypertension,33.959893,5.756201e+09
6,2019,Obesity,14.054000,1.664804e+08
7,2019,Other / Not classified,237.540063,1.276343e+11
8,2020,Arthritis,1523.295965,2.505404e+10
9,2020,Arthritis;Hypertension,6.482878,2.228778e+08


In [15]:
# access specifc yearly data
medicare_conditon_agg[medicare_conditon_agg['Year'] == 2019]

,Year,Disease_Category,Avg_Price_By_Cond,Total_Spend_By_Cond
0,2019,Arthritis,1543.157217,2.198443e+10
1,2019,Arthritis;Hypertension,6.324735,2.472512e+08
2,2019,Diabetes,59.982048,1.766225e+10
3,2019,Diabetes;High Cholesterol,0.138261,2.069756e+08
4,2019,High Cholesterol,122.985532,3.091740e+09
5,2019,Hypertension,33.959893,5.756201e+09
6,2019,Obesity,14.054000,1.664804e+08
7,2019,Other / Not classified,237.540063,1.276343e+11


In [23]:
medicare_breakdown = medicare_with_cond[medicare_with_cond['Mftr_Name'] != 'Overall']

# clean astrik
medicare_breakdown['Mftr_Name'] = (
    medicare_breakdown['Mftr_Name'].str.replace(r'\*', '', regex=True)
)

medicare_breakdown.head()

/var/folders/5b/6lvktnb96955q0486jnqvwf80000gn/T/ipykernel_74899/1636339457.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  medicare_breakdown['Mftr_Name'] = (


,Brnd_Name,Gnrc_Name,Tot_Mftr,Mftr_Name,Year,CAGR_Avg_Spnd_Per_Dsg_Unt_19_23,Chg_Avg_Spnd_Per_Dsg_Unt_22_23,Avg_Spnd_Per_Bene,Avg_Spnd_Per_Clm,Avg_Spnd_Per_Dsg_Unt_Wghtd,Outlier_Flag,Tot_Benes,Tot_Clms,Tot_Dsg_Unts,Tot_Spndng,ATC_Codes,Disease_Category
5,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Owen Mumford Us,2019,0.011754,0.005702,74.122300,25.816335,0.216788,0.0,1878.0,5392.0,642471.0,139201.68,NaN,Other / Not classified
6,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Owen Mumford Us,2020,0.011754,0.005702,74.560025,26.682351,0.217701,0.0,1595.0,4457.0,547006.0,118923.24,NaN,Other / Not classified
7,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Owen Mumford Us,2021,0.011754,0.005702,77.898522,27.583808,0.223001,0.0,1313.0,3708.0,459384.0,102280.76,NaN,Other / Not classified
8,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Owen Mumford Us,2022,0.011754,0.005702,61.063304,28.004642,0.225874,0.0,1147.0,2501.0,310304.0,70039.61,NaN,Other / Not classified
9,1st Tier Unifine Pentips,"Pen Needle, Diabetic",1,Owen Mumford Us,2023,0.011754,0.005702,63.454993,27.498475,0.227162,0.0,699.0,1613.0,195672.0,44355.04,NaN,Other / Not classified


In [24]:
# check cleaning success 
medicare_breakdown[medicare_breakdown['Mftr_Name'].str.contains(r'\*')]

,Brnd_Name,Gnrc_Name,Tot_Mftr,Mftr_Name,Year,CAGR_Avg_Spnd_Per_Dsg_Unt_19_23,Chg_Avg_Spnd_Per_Dsg_Unt_22_23,Avg_Spnd_Per_Bene,Avg_Spnd_Per_Clm,Avg_Spnd_Per_Dsg_Unt_Wghtd,Outlier_Flag,Tot_Benes,Tot_Clms,Tot_Dsg_Unts,Tot_Spndng,ATC_Codes,Disease_Category


### Test Visualizations

In [ ]:
PHARMA = 'Eli Lilly & Co.'
df_pfizer = df_long[df_long['Mftr_Name'] == PHARMA]
df_pfizer

In [ ]:
# --- to-do: add in filter so only see top 10 
fig = px.line(
    df_pfizer.sort_values(by='Year'),
    x='Year',
    y='Tot_Spndng',
    color='Brnd_Name', 
    markers=True,   
    title=f'Total Spending for {PHARMA} by Brand (2019-2023)'
)

fig.show()

In [ ]:
# --- drop all nan values for plotting purposes 
df_plot = df_long.dropna()
df_plot.head()

In [ ]:
# to-do's: --------- turn this interactive somehow (maybe make this Tableau?)
N = 20

df_filter = df_plot[df_plot['Mftr_Name'] != 'Overall']

# (Optional) filter top manufacturers by total spending
top_mfrs = (
    df_filter.groupby("Mftr_Name")["Tot_Spndng"]
    .sum()
    .nlargest(N)
    .index
)
df_top = df_filter[df_filter["Mftr_Name"].isin(top_mfrs)]

# Create the treemap
fig = px.treemap(
    df_top,
    path=["Mftr_Name", "Brnd_Name"],     # hierarchy: manufacturer → brand
    values="Tot_Spndng",                # rectangle size = total spending
    color="Avg_Spnd_Per_Dsg_Unt_Wghtd",             # color = average unit price
    color_continuous_scale="Viridis",   # any Plotly color scale
    hover_data={
        "Tot_Spndng": ":,.0f",          # show formatted spending
        "Avg_Spnd_Per_Dsg_Unt_Wghtd": ":,.2f",
        "Outlier_Flag": True
    },
    title=f"Drug Price Differences by Manufacturer (Top {N} Manufacturers)"
)

# Optional layout adjustments
fig.update_layout(
    title_font_size=18,
    margin=dict(t=50, l=25, r=25, b=25)
)

fig.show()

In [ ]:
# experiment with filtering by condition within the gnrc_name
conditions = ['diabetic', 'diabeties', 'cancer']

df_conditions = df_long[df_long['Gnrc_Name']
                      .str.contains('|'.join(conditions), case=False, na=False)]
df_conditions

In [ ]:
set(df_conditions['Gnrc_Name'].to_list())

In [ ]:
# Merge atc_disease_mapping to medicare_drug_spending

# 1. Load original Medicare file
medicare = pd.read_csv("medicare_drug_spending.csv")

# 2. Load mapping file
mapping = pd.read_csv("atc_disease_mapping.csv")


In [ ]:
# Check to confirm columns
print(medicare.columns[:5])


In [ ]:
print(mapping.head())

In [ ]:
# 3. Merge on Gnrc_Name (generic name)
medicare_with_cond = medicare.merge(
    mapping[["Gnrc_Name", "ATC_Codes", "Disease_Category"]],
    on="Gnrc_Name",
    how="left"    # keep all Medicare rows, add condition info where available
)

# 4. Make blanks explicit by labelling as "Other / Not classified"
medicare_with_cond["Disease_Category"] = (
    medicare_with_cond["Disease_Category"]
    .fillna("Other / Not classified")
)

# 5. Save the enriched dataset
medicare_with_cond.to_csv("medicare_with_conditions.csv", index=False)

print("Saved medicare_with_conditions.csv with shape:", medicare_with_cond.shape)
medicare_with_cond.head()